In [1]:
# Import libraries for modeling, text processing, and validation

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier  # Optimized Gradient Boosting classifier

# Load the '20 Newsgroups' dataset, a collection of English texts classified by topic

data = fetch_20newsgroups(subset='all', remove=())  # The full dataset is loaded without removing any part

# Create a DataFrame with the document text and its numeric label
df = pd.DataFrame({
    'text': data.data,
    'original_label': data.target
})

# Add a column with the name of the textual category corresponding to each label
category_names = data.target_names
df['topic'] = df['original_label'].apply(lambda x: category_names[x])

# Definition of a binary variable "sport"
# Positive classes: categories related to sports and motor vehicles

sport_categories = ['rec.sport.baseball', 'rec.sport.hockey', 'rec.autos', 'rec.motorcycles']
df['sport'] = df['topic'].apply(lambda x: 1 if x in sport_categories else 0)

# Split the data into features (X) and binary labels (y)

X = df['text'].values
y = df['sport'].values

# Definition of a custom stopword list
# These words will be ignored during vectorization

stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]


In [2]:
i=0
# Initialization of cross-validation
# The dataset is divided into 5 partitions (folds)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []  # List to store the AUC score of each fold
# Train and evaluate the model for each fold

for train_idx, test_idx in cv.split(X, y):
    # Split the data into training and test sets
    X_train_raw, X_test_raw = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Vectorize the text with TF-IDF
    # The vectorizer is fitted only on the training set
    # to avoid any information leakage (data leakage)

    vectorizer = TfidfVectorizer(max_features=10000, stop_words=stopwords)
    X_train = vectorizer.fit_transform(X_train_raw)
    X_test = vectorizer.transform(X_test_raw)


    # Train the XGBoost model (Gradient Boosting for binary classification)
    # Common parameters: maximum depth, number of trees, metric, and seed


    model = XGBClassifier(
        max_depth=6,
        n_estimators=500,  
        use_label_encoder=False, # Disabled to avoid old warnings
        eval_metric='logloss',
        random_state=42
    )
    model.fit(X_train, y_train)

    print("Iteration: "+str(i) ) # Progress message
    i=i+1

    # Predict probabilities for the positive class (1)
    # and compute the AUC-ROC metric on the test set

    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    print("AUC-ROC: Fold " + str(i) +  ":" + str(auc))  # Print the AUC score of the current fold

    scores.append(auc)  # Add the result of the current fold


Iteration: 0
AUC-ROC: Fold 1:0.9914700435601005


Iteration: 1
AUC-ROC: Fold 2:0.9903283656763467


Iteration: 2
AUC-ROC: Fold 3:0.9891287077837895


Iteration: 3
AUC-ROC: Fold 4:0.9942759542752444


Iteration: 4
AUC-ROC: Fold 5:0.9922045568934963


In [3]:
# Organize the results
results=[]
results.append({
        'k_components': 10000,
        'mean_auc_roc': np.mean(scores)
    })

df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=10000):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=10000):

 k_components  mean_auc_roc
        10000        0.9915
